# Baseline Churn Model Check + Initial Behavioral Drift Score

Input: `data/features_and_label.parquet` (customer-month rows from notebook 03, label `is_churn`)

Output: drift features + `drift_score` per customer-month

1. Verify that the baseline churn models from 06 beat trivial baselines and are not fooled by leakage
2. Build the first behavioral-drift score & test whether it carries churn signal beyond th existing features


## 1. Setup + Load Data

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

In [4]:
DATA_PATH     = Path('../data/features_and_label.parquet')
OUT_PATH      = Path('../data/drift_scores.parquet')
TARGET        = 'is_churn'

TEST_FRACTION = 0.20   # same as notebook 06: latest 20% of distinct months are the test set

PURGE_MONTHS  = 3      # = the 90-day label window (~3 months)

K_BASELINE    = 3      # a customer's "normal" = mean of their previous 3 ACTIVE months
CLIP_Z        = 3.0    # cap each standardized drift component at +/-3 so one freak month can't dominate
RANDOM_STATE  = 42

In [5]:
df = pd.read_parquet(DATA_PATH)

# CustomerMonth is stored as a Period[M], so we'll convert to a month-start timestamp (same handling as notebook 06)
cm = df['CustomerMonth']
df['CustomerMonth'] = cm.dt.to_timestamp() if isinstance(cm.dtype, pd.PeriodDtype) else pd.to_datetime(cm)

df = df.dropna(subset=[TARGET, 'CustomerMonth']).copy()
df[TARGET] = df[TARGET].astype(int)
df['is_first_month'] = df['is_first_month'].astype(int)   # bool -> 0/1 so every model treats it as a plain number

# sort by customer then month
# all "previous months" logic below (shift/rolling) depends on this order
df = df.sort_values(['Customer ID', 'CustomerMonth']).reset_index(drop=True)

print(f'Rows: {len(df):,} | customers: {df["Customer ID"].nunique():,} | churn rate: {df[TARGET].mean():.2%}')
print(f'Months: {df.CustomerMonth.min():%Y-%m} to {df.CustomerMonth.max():%Y-%m}')
display(df.head())

Rows: 20,880 | customers: 5,248 | churn rate: 38.99%
Months: 2009-12 to 2011-09


,Customer ID,CustomerMonth,purchase_frequency,total_spend,avg_order_value,unique_items,days_since_last_purchase,is_first_month,is_churn
0,12346,2010-03-01,1,27.05,27.05,5,NaN,1,1
1,12346,2010-06-01,1,142.31,142.31,19,118.0,0,1
2,12346,2011-01-01,1,149.76,149.76,1,203.0,0,1
3,12347,2010-10-01,1,611.53,611.53,40,NaN,1,0
4,12347,2010-12-01,1,711.79,711.79,31,37.0,0,0
